In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Smart mcq solver**

## milestone 0

In [2]:
%pip install -q transformers datasets accelerate scikit-learn tqdm sentencepiece protobuf gensim ## Install required libraries


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import all necessary packages

import os
import re
import glob
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pandas as pd
import numpy as np

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMultipleChoice,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore") # hide warning messages for cleaner output


import sentencepiece #needed for some transformer models (DeBERTa uses SentencePiece tokenization)

os.environ["TOKENIZERS_PARALLELISM"] = "false" #disable tokenizer parallelism warnings

In [ ]:
 # Set random seeds for reproducibility

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Check what computing device is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device name ", device)


In [ ]:
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

DEBERTA_MODEL_PATH = "microsoft/deberta-v3-base"
DISTILBERT_MODEL_PATH = "distilbert-base-uncased"

SCRATCH_MODEL_WEIGHTS = "best_model_scratch.pt" ## saved model weights 


# training flags
RUN_TRAINING_FROM_SCRATCH = False
RUN_TRANSFORMER_TRAINING = True

# ANN hyperparameters
ANN_EPOCHS = 5
ANN_BATCH_SIZE = 32
ANN_LR = 0.001
ANN_EMBED_DIM = 256
ANN_HIDDEN_DIM = 256

# Transformer Hyperparameters 
TRANSFORMER_EPOCHS = 10
TRANSFORMER_BATCH_SIZE = 2
DEBERTA_LR = 5e-6
DISTILBERT_LR = 2e-5
GRAD_ACCUM_STEPS = 8

# ensemble threshold 
MIN_USABLE_MAP3 = 0.40

LABEL_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}



In [ ]:
# Load the datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Basic dataset information
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

# Find the answer column
LABEL_COL = None

for col in ["answer", "Answer", "label", "Label", "correct_answer"]:
    if col in train_df.columns:
        LABEL_COL = col
        break

if LABEL_COL:
    print("Answer column:", LABEL_COL)
    print(train_df[LABEL_COL].value_counts())
else:
    print("No answer column found.")
    RUN_TRAINING_FROM_SCRATCH = False
    RUN_TRANSFORMER_TRAINING = False

In [ ]:
train_df.head(2)

In [ ]:
# Remove duplicate questions
train_df_clean = train_df.drop_duplicates(
    subset=["prompt", "A", "B", "C", "D", "E"]
)

print("Original dataset size:", len(train_df))
print("After removing duplicates:", len(train_df_clean))
print("Duplicates removed:", len(train_df) - len(train_df_clean))

In [ ]:
# split the data into training and validation sets
gss = GroupShuffleSplit(test_size=0.1, n_splits=1, random_state=42)

train_idx, val_idx = next(
    gss.split(train_df, groups=train_df["prompt"])
)

t_df = train_df.iloc[train_idx]
v_df = train_df.iloc[val_idx]

print("Training samples:", len(t_df))
print("Validation samples:", len(v_df))


## Milestone 1 — (NLP Foundation & Semantic Similarity)

In [ ]:
# Clean text by converting to lowercase and removing special characters
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
# Check for missing data
print(train_df.isnull().sum())
print(test_df.isnull().sum())

In [ ]:
# Calculate mAP@3 score
def calculate_map3(logits, targets):
    top3 = np.argsort(logits, axis=1)[:, ::-1][:, :3]

    scores = []
    for pred, true in zip(top3, targets):
        if true == pred[0]:
            scores.append(1.0)
        elif true == pred[1]:
            scores.append(0.5)
        elif true == pred[2]:
            scores.append(1 / 3)
        else:
            scores.append(0.0)

    return np.mean(scores)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build TF-IDF vocabulary
all_text = []
for col in ["prompt", "A", "B", "C", "D", "E"]:
    all_text.extend(train_df[col].astype(str).apply(clean_text))

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)
tfidf_vectorizer.fit(all_text)

print("TF-IDF vocabulary size:", len(tfidf_vectorizer.vocabulary_))

# Predict answers using cosine similarity
def tfidf_predict(df, vectorizer):
    predictions = []

    for _, row in df.iterrows():
        prompt_vec = vectorizer.transform([clean_text(row["prompt"])])

        scores = []
        for option in ["A", "B", "C", "D", "E"]:
            option_vec = vectorizer.transform([clean_text(row[option])])
            scores.append(cosine_similarity(prompt_vec, option_vec)[0][0])

        predictions.append(scores)

    return np.array(predictions)

tfidf_preds = tfidf_predict(v_df, tfidf_vectorizer)
tfidf_targets = v_df[LABEL_COL].map(LABEL_MAP).values

tfidf_map3 = calculate_map3(tfidf_preds, tfidf_targets)

print("TF-IDF mAP@3:", round(tfidf_map3, 4))

In [ ]:
from gensim.models import Word2Vec

# Prepare text for Word2Vec
sentences = []

for col in ["prompt", "A", "B", "C", "D", "E"]:
    for text in train_df[col].astype(str):
        tokens = clean_text(text).split()
        if tokens:
            sentences.append(tokens)

# Train the Word2Vec model
w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10,
    seed=42
)

print("Vocabulary size:", len(w2v_model.wv))

# Convert text into a vector
def text_to_w2v_vector(text, model):
    tokens = clean_text(text).split()
    vectors = [model.wv[token] for token in tokens if token in model.wv]

    if vectors:
        return np.mean(vectors, axis=0)

    return np.zeros(model.vector_size)

# Predict answers using cosine similarity
def w2v_predict(df, model):
    predictions = []

    for _, row in df.iterrows():
        prompt_vec = text_to_w2v_vector(row["prompt"], model).reshape(1, -1)

        scores = []
        for option in ["A", "B", "C", "D", "E"]:
            option_vec = text_to_w2v_vector(row[option], model).reshape(1, -1)
            scores.append(cosine_similarity(prompt_vec, option_vec)[0][0])

        predictions.append(scores)

    return np.array(predictions)

w2v_preds = w2v_predict(v_df, w2v_model)
w2v_targets = v_df[LABEL_COL].map(LABEL_MAP).values

w2v_map3 = calculate_map3(w2v_preds, w2v_targets)

print("Word2Vec mAP@3:", round(w2v_map3, 4))